# 🚁 SUTRA — Subsystem C: VisDrone Fine-Tuning
**Project:** SUTRA (Swarm Unified Tactical Reconnaissance Architecture)  
**Lead:** Vedanth Sai Ram — Subsystem C (AI Perception)  
**Goal:** Fine-tune YOLOv8-Nano on VisDrone2019 aerial dataset  
**Target:** mAP@0.5 ≥ 90% on aerial drone imagery  
**Gate:** G3 — Edge AI Survivor Perception  

---

**Steps:**
1. Install dependencies
2. Download VisDrone2019-DET dataset (~2.5GB)
3. Fine-tune YOLOv8-Nano for 50 epochs on T4 GPU
4. Validate — report mAP@0.5
5. Download best.pt weights for Jetson TensorRT export


In [ ]:
# ── Step 1: Install dependencies ──────────────────────────────────────────────
!pip install ultralytics==8.2.0 -q

import os, time, json, shutil
import numpy as np
from pathlib import Path
from ultralytics import YOLO
import torch

print(f'✅ PyTorch version : {torch.__version__}')
print(f'✅ CUDA available  : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'✅ GPU             : {torch.cuda.get_device_name(0)}')
    print(f'✅ VRAM            : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')


In [ ]:
# ── Step 2: Download VisDrone2019-DET dataset ─────────────────────────────────
# VisDrone2019: 10,209 drone aerial images, 543,000+ person labels
# Official source: DroneVision Group / Ultralytics mirror

import urllib.request, zipfile

BASE_DIR = Path('/kaggle/working/visdrone')
BASE_DIR.mkdir(exist_ok=True)

SPLITS = {
    'VisDrone2019-DET-train': 'https://github.com/ultralytics/assets/releases/download/v0.0.0/VisDrone2019-DET-train.zip',
    'VisDrone2019-DET-val'  : 'https://github.com/ultralytics/assets/releases/download/v0.0.0/VisDrone2019-DET-val.zip',
    'VisDrone2019-DET-test-dev': 'https://github.com/ultralytics/assets/releases/download/v0.0.0/VisDrone2019-DET-test-dev.zip',
}

for name, url in SPLITS.items():
    zip_path = BASE_DIR / f'{name}.zip'
    if not zip_path.exists():
        print(f'⬇  Downloading {name}...')
        urllib.request.urlretrieve(url, zip_path)
        print(f'📦 Extracting {name}...')
        with zipfile.ZipFile(zip_path, 'r') as z:
            z.extractall(BASE_DIR)
        print(f'✅ {name} ready')
    else:
        print(f'✅ {name} already downloaded')

# Count images
train_imgs = list((BASE_DIR / 'VisDrone2019-DET-train' / 'images').glob('*.jpg'))
val_imgs   = list((BASE_DIR / 'VisDrone2019-DET-val'   / 'images').glob('*.jpg'))
print(f'\n📊 Train images : {len(train_imgs)}')
print(f'📊 Val images   : {len(val_imgs)}')


In [ ]:
# ── Step 3: Convert VisDrone annotations → YOLO format ───────────────────────
# VisDrone uses: x, y, w, h, score, category, truncation, occlusion
# YOLO needs: class cx cy w h (all normalised 0-1)
#
# VisDrone classes → mapped to SUTRA SAR-relevant classes:
#   0=ignored, 1=pedestrian(→person), 2=people(→person), 3=bicycle,
#   4=car, 5=van, 6=truck, 7=tricycle, 8=awning-tricycle,
#   9=bus, 10=motor, 11=others

VISDRONE_TO_YOLO = {
    1: 0,   # pedestrian → person
    2: 0,   # people     → person
    3: 1,   # bicycle
    4: 2,   # car
    5: 3,   # van
    6: 4,   # truck
    7: 5,   # tricycle
    8: 5,   # awning-tricycle
    9: 6,   # bus
    10:7,   # motor
}

CLASS_NAMES = ['person', 'bicycle', 'car', 'van', 'truck', 'tricycle', 'bus', 'motor']

def convert_visdrone_to_yolo(ann_dir: Path, img_dir: Path, out_dir: Path):
    out_dir.mkdir(parents=True, exist_ok=True)
    converted, skipped = 0, 0
    for ann_file in ann_dir.glob('*.txt'):
        img_file = img_dir / ann_file.name.replace('.txt', '.jpg')
        if not img_file.exists():
            skipped += 1; continue

        import cv2
        img = cv2.imread(str(img_file))
        if img is None: skipped += 1; continue
        H, W = img.shape[:2]

        yolo_lines = []
        for line in ann_file.read_text().strip().split('\n'):
            parts = line.strip().split(',')
            if len(parts) < 6: continue
            x, y, w, h, score, cat = int(parts[0]), int(parts[1]), int(parts[2]), int(parts[3]), int(parts[4]), int(parts[5])
            if w <= 0 or h <= 0 or cat == 0 or cat not in VISDRONE_TO_YOLO: continue
            yolo_cls = VISDRONE_TO_YOLO[cat]
            cx = (x + w/2) / W
            cy = (y + h/2) / H
            nw = w / W
            nh = h / H
            yolo_lines.append(f'{yolo_cls} {cx:.6f} {cy:.6f} {nw:.6f} {nh:.6f}')

        (out_dir / ann_file.name).write_text('\n'.join(yolo_lines))
        converted += 1

    print(f'  Converted: {converted} | Skipped: {skipped}')

print('Converting train annotations...')
convert_visdrone_to_yolo(
    BASE_DIR / 'VisDrone2019-DET-train' / 'annotations',
    BASE_DIR / 'VisDrone2019-DET-train' / 'images',
    BASE_DIR / 'VisDrone2019-DET-train' / 'labels'
)

print('Converting val annotations...')
convert_visdrone_to_yolo(
    BASE_DIR / 'VisDrone2019-DET-val' / 'annotations',
    BASE_DIR / 'VisDrone2019-DET-val' / 'images',
    BASE_DIR / 'VisDrone2019-DET-val' / 'labels'
)
print('✅ Annotation conversion complete')


In [ ]:
# ── Step 4: Write dataset YAML ────────────────────────────────────────────────

YAML_PATH = Path('/kaggle/working/visdrone_sutra.yaml')
YAML_PATH.write_text(f"""
# SUTRA VisDrone2019 Dataset Config
path: {BASE_DIR}
train: VisDrone2019-DET-train/images
val:   VisDrone2019-DET-val/images
test:  VisDrone2019-DET-test-dev/images

nc: {len(CLASS_NAMES)}
names: {CLASS_NAMES}

# SUTRA SAR Focus: class 0 (person) is the primary survivor target
""")

print(f'✅ Dataset YAML written to {YAML_PATH}')
print(YAML_PATH.read_text())


In [ ]:
# ── Step 5: Fine-tune YOLOv8-Nano on VisDrone ────────────────────────────────
# T4 GPU: ~50 epochs takes ~90-120 minutes

model = YOLO('yolov8n.pt')  # Start from COCO pre-trained weights

print('🚀 Starting VisDrone fine-tuning...')
print(f'   Base model : YOLOv8-Nano (COCO pre-trained)')
print(f'   Dataset    : VisDrone2019-DET ({len(train_imgs)} train, {len(val_imgs)} val)')
print(f'   Epochs     : 50')
print(f'   Image size : 640x640')
print(f'   GPU        : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}')
print()

t0 = time.time()

results = model.train(
    data      = str(YAML_PATH),
    epochs    = 50,
    imgsz     = 640,
    batch     = 16,           # T4 GPU can handle 16
    patience  = 15,           # early stop if no improvement for 15 epochs
    device    = 0,            # GPU
    workers   = 4,
    project   = '/kaggle/working/sutra_train',
    name      = 'yolov8n_visdrone',
    exist_ok  = True,
    pretrained= True,
    optimizer = 'AdamW',
    lr0       = 0.001,
    lrf       = 0.01,
    mosaic    = 1.0,          # data augmentation — key for aerial images
    mixup     = 0.1,
    degrees   = 15.0,         # rotation augmentation for aerial
    flipud    = 0.5,          # vertical flip (aerial images can be any orientation)
    fliplr    = 0.5,
    scale     = 0.5,          # scale jitter
    translate = 0.1,
    save      = True,
    save_period = 10,         # save checkpoint every 10 epochs
    val       = True,
    plots     = True,
    verbose   = True,
)

elapsed = time.time() - t0
print(f'\n✅ Training complete in {elapsed/60:.1f} minutes')


In [ ]:
# ── Step 6: Validate best model — report mAP@0.5 ─────────────────────────────

best_model_path = '/kaggle/working/sutra_train/yolov8n_visdrone/weights/best.pt'
best_model = YOLO(best_model_path)

print('🔍 Running validation on VisDrone val set...')
metrics = best_model.val(
    data    = str(YAML_PATH),
    device  = 0,
    verbose = True,
)

map50    = metrics.box.map50
map50_95 = metrics.box.map
mp       = metrics.box.mp
mr       = metrics.box.mr

# Person class specifically (class 0)
person_ap50 = metrics.box.ap50[0] if len(metrics.box.ap50) > 0 else 0.0

print(f"""
╔══════════════════════════════════════════════════════════╗
║   🚁  SUTRA SUBSYSTEM C — GATE G3 RESULTS               ║
╠══════════════════════════════════════════════════════════╣
║  mAP@0.5 (all classes)  : {map50*100:>6.2f}%                   ║
║  mAP@0.5:0.95           : {map50_95*100:>6.2f}%                   ║
║  Person AP@0.5          : {person_ap50*100:>6.2f}%  ← SAR critical   ║
║  Mean Precision         : {mp*100:>6.2f}%                   ║
║  Mean Recall            : {mr*100:>6.2f}%                   ║
╠══════════════════════════════════════════════════════════╣
║  Gate G3 target: mAP@0.5 >= 90%                         ║
║  Result: {'✅ PASSED' if map50 >= 0.9 else '⚠  ' + str(round(map50*100,1)) + '% — close but below 90%':<42}  ║
╚══════════════════════════════════════════════════════════╝
""")

# Save results JSON
result_json = {
    'model'        : 'yolov8n_visdrone_finetuned',
    'dataset'      : 'VisDrone2019-DET',
    'epochs'       : 50,
    'map50'        : round(float(map50), 4),
    'map50_95'     : round(float(map50_95), 4),
    'person_ap50'  : round(float(person_ap50), 4),
    'precision'    : round(float(mp), 4),
    'recall'       : round(float(mr), 4),
    'gate_g3_pass' : bool(map50 >= 0.9),
    'subsystem'    : 'C — AI Perception',
    'lead'         : 'Vedanth Sai Ram',
}
Path('/kaggle/working/gate_g3_results.json').write_text(json.dumps(result_json, indent=2))
print('✅ Results saved to /kaggle/working/gate_g3_results.json')


In [ ]:
# ── Step 7: Test inference on sample VisDrone images ──────────────────────────

import cv2, matplotlib.pyplot as plt

sample_imgs = list((BASE_DIR / 'VisDrone2019-DET-val' / 'images').glob('*.jpg'))[:4]

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('SUTRA YOLOv8-Nano Fine-tuned on VisDrone — Sample Detections', fontsize=14, fontweight='bold')

for i, (img_path, ax) in enumerate(zip(sample_imgs, axes.flat)):
    res = best_model(str(img_path), conf=0.25, verbose=False)[0]
    annotated = res.plot()
    annotated_rgb = cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB)
    persons = sum(1 for c in res.boxes.cls.tolist() if int(c) == 0)
    ax.imshow(annotated_rgb)
    ax.set_title(f'Image {i+1} — {len(res.boxes)} detections ({persons} persons)', fontsize=11)
    ax.axis('off')

plt.tight_layout()
plt.savefig('/kaggle/working/sample_detections.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Sample detection image saved')


In [ ]:
# ── Step 8: Export to ONNX (runs on Jetson + CPU) ────────────────────────────
# Note: Full TensorRT .engine export needs to be done ON the Jetson itself
# We export ONNX here which can then be converted to TensorRT on Jetson

print('📦 Exporting to ONNX format (Jetson-compatible)...')
best_model.export(
    format    = 'onnx',
    imgsz     = 640,
    half      = False,   # set True on Jetson for FP16
    dynamic   = False,
    simplify  = True,
)
print('✅ ONNX export done: best.onnx')

# List all output files
print('\n📁 Output files for download:')
for f in Path('/kaggle/working/sutra_train/yolov8n_visdrone/weights').glob('*'):
    size_mb = f.stat().st_size / 1e6
    print(f'  {f.name:<25} {size_mb:.1f} MB')

print('\n📁 Results:')
print('  gate_g3_results.json')
print('  sample_detections.png')


In [ ]:
# ── Step 9: Final summary ─────────────────────────────────────────────────────

print(f"""
╔══════════════════════════════════════════════════════════════╗
║   🚁  SUTRA SUBSYSTEM C — TRAINING COMPLETE                 ║
╠══════════════════════════════════════════════════════════════╣
║  Model    : YOLOv8-Nano fine-tuned on VisDrone2019          ║
║  mAP@0.5  : {map50*100:.1f}%  (Gate G3 target: >= 90%)         ║
║  Person   : {person_ap50*100:.1f}% AP  (SAR critical class)          ║
╠══════════════════════════════════════════════════════════════╣
║  DOWNLOAD these files:                                      ║
║  1. sutra_train/yolov8n_visdrone/weights/best.pt            ║
║     → Replace yolov8n.pt in detector_node.py                ║
║  2. sutra_train/yolov8n_visdrone/weights/best.onnx          ║
║     → Convert to TensorRT .engine on Jetson                 ║
║  3. gate_g3_results.json                                    ║
║     → Share with Harika for Gate G3 audit                   ║
║  4. sample_detections.png                                   ║
║     → Use in hackathon presentation                         ║
╠══════════════════════════════════════════════════════════════╣
║  On Jetson: convert ONNX → TensorRT                         ║
║  trtexec --onnx=best.onnx --saveEngine=best.engine          ║
║           --fp16 --workspace=1024                           ║
╚══════════════════════════════════════════════════════════════╝
""")
